### A Student's Guide to Torchvision Models

Torchvision is a PyTorch package that provides access to:
- popular datasets,
- model architectures,
- and image transformations for computer vision.

It simplifies the process of building image-based deep learning projects by offering pre-trained models, ready for use or fine-tuning.

Why use pre-trained models becasue it enables:

    Faster training: Start with learned features instead of random initialization.
    Less data required: Benefit from models already trained on massive datasets.
    Higher accuracy: Utilize architectures optimized for image tasks.

The core componens of Torchvison are:

- __torchvision.models:__ Provides a collection of popular model architectures like ResNet, AlexNet, VGG, and more.
- __torchvision.datasets:__ Offers access to common datasets such as ImageNet, CIFAR10, MNIST, etc.
- __torchvision.transforms:__ Includes tools for image manipulation (resizing, cropping, normalization).

### Using Pre-trained Models

In [1]:
# The usual libraries required from the pytorch packages

import torch
import torchvision
from torchvision import models, transforms
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt

- The *torchvision.models* module has a host of pretrained models for vision applications (image classification, etc).
- These pretrained models all have a history of how it was developed
- There are also benchmarks and this is usually available as published paper.

As an example: The class of Resnet models can be understood in details by reading this paper - https://paperswithcode.com/method/resnet

In [4]:
import pprint

In [8]:
# let us look at the available pretrained models
# We will see there is a long list

# Get a list of all available models
model_list = models.list_models()
print(f"List of available pretrained models in the tochvision package:\n\n{model_list}")

List of available pretrained models in the tochvision package:

['alexnet', 'convnext_base', 'convnext_large', 'convnext_small', 'convnext_tiny', 'deeplabv3_mobilenet_v3_large', 'deeplabv3_resnet101', 'deeplabv3_resnet50', 'densenet121', 'densenet161', 'densenet169', 'densenet201', 'efficientnet_b0', 'efficientnet_b1', 'efficientnet_b2', 'efficientnet_b3', 'efficientnet_b4', 'efficientnet_b5', 'efficientnet_b6', 'efficientnet_b7', 'efficientnet_v2_l', 'efficientnet_v2_m', 'efficientnet_v2_s', 'fasterrcnn_mobilenet_v3_large_320_fpn', 'fasterrcnn_mobilenet_v3_large_fpn', 'fasterrcnn_resnet50_fpn', 'fasterrcnn_resnet50_fpn_v2', 'fcn_resnet101', 'fcn_resnet50', 'fcos_resnet50_fpn', 'googlenet', 'inception_v3', 'keypointrcnn_resnet50_fpn', 'lraspp_mobilenet_v3_large', 'maskrcnn_resnet50_fpn', 'maskrcnn_resnet50_fpn_v2', 'maxvit_t', 'mc3_18', 'mnasnet0_5', 'mnasnet0_75', 'mnasnet1_0', 'mnasnet1_3', 'mobilenet_v2', 'mobilenet_v3_large', 'mobilenet_v3_small', 'mvit_v1_b', 'mvit_v2_s', 'quantiz

#### Example of loading a pretrained model and looking at its architecture and its layers

In [10]:
# Load a ResNet18 model with pre-trained ImageNet weights
# This model was trained originally on 1000 classes and hence has 1000 outputs

# Get the available weights for ResNet18
weights = models.get_model_weights("resnet18")

# Get the model and load it with the weights
model = models.resnet18(weights=weights) 
#print(f"The architecture of the model has many convolution layers and a fully connected classifier at the end:\n\n{model}")

In [11]:
# These models have been trained with different training parameters and hence have different version of weights
# The ".DEFAULT" parameter gives us  = best available weights from pretraining and is always a safe bet

# Get the default weights for ResNet18
default_weights = models.ResNet18_Weights.DEFAULT

# Get the model and apply the weights to it
model = models.resnet18(weights=default_weights)

In [12]:
# The models are all pretrained on 1000s of images and these images were preprocessed (transforms were applied on them).
# When we use these models for transfer learning, we should do the same preprocessing on our custom image dataset

# There is a convenient way of getting the transforms that were done on the original images
# We get it from the weights information

auto_transforms = default_weights.transforms()
print(f"Originally used transforms are:\n{auto_transforms}")

Originally used transforms are:
ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


#### We can also scan the model layers and access them - for freezing layers, changing the output classes, etc

- model.named_children(): This method provides an iterator that yields pairs of (name, layer) for each direct child of the model.
- for loop: The loop iterates through these pairs, allowing you to access the name and the layer object itself.
- print(): You can print the name to see what the layer is called and type(layer) to see its type (e.g., torch.nn.Conv2d, torch.nn.ReLU, torch.nn.Linear).
- Further Inspection: You can optionally print the layer object itself to see its detailed configuration (kernel size, stride, etc.).

In [13]:
# We run a for loop and go layer by layer to get the names of the layers

# Iterate through the model's children (layers)
for name, layer in model.named_children():
    print(f"Layer name: {name}")
    print(f"Layer type: {type(layer)}")
    # You can further inspect layer attributes if needed
    #print(layer)  # This will print the layer's configuration details

Layer name: conv1
Layer type: <class 'torch.nn.modules.conv.Conv2d'>
Layer name: bn1
Layer type: <class 'torch.nn.modules.batchnorm.BatchNorm2d'>
Layer name: relu
Layer type: <class 'torch.nn.modules.activation.ReLU'>
Layer name: maxpool
Layer type: <class 'torch.nn.modules.pooling.MaxPool2d'>
Layer name: layer1
Layer type: <class 'torch.nn.modules.container.Sequential'>
Layer name: layer2
Layer type: <class 'torch.nn.modules.container.Sequential'>
Layer name: layer3
Layer type: <class 'torch.nn.modules.container.Sequential'>
Layer name: layer4
Layer type: <class 'torch.nn.modules.container.Sequential'>
Layer name: avgpool
Layer type: <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>
Layer name: fc
Layer type: <class 'torch.nn.modules.linear.Linear'>


What we can see from the above is:
- the models has a conv1 layer as the input layer
- then has 4 convolution layers
- and finally a fully connected (fc) layer which the classifier part of the model.

##### We can now choose to freeze all convolution layers to put our model into 'feature extraction' mode

In [19]:
# We freeze all the parameters of the model by saying "requires_grad=False". This would include all the weights of convolution and
# fully connected (fc) layers (so the whole model).

# However, we want to freeze only the convolution layers, but, when we modify the outputs of the fc layers, 
# by default its requires_grad is set to True

for param in model.parameters():
    param.requires_grad = False

# Modify the final fully connected layer for your task - that is as many output or classes out application will have.
# Suppose we had 4 output
num_classes = 4

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)  # num_classes is the number of classes in your task

print(f"Check output is modified: {model.fc}\nVerify  if fc layer has requires_grad set to true: {model.fc.weight.requires_grad}")

Check output is modified: Linear(in_features=512, out_features=4, bias=True)
Verify  if fc layer has requires_grad set to true: True


##### Freeze only the layers we want to freeze (as an example say all but the last 2 convolution layers)

In [20]:
# Freeze all layers except the last two
for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:  # Check if the parameter belongs to the last two layers
        param.requires_grad = True  # Unfreeze these layers
    else:
        param.requires_grad = False  # Freeze all other layers

#### Using the torchinfo summary module to check the status of the layers

You can install torchinfo with 'pip install torchinfo'

In [21]:
from torchinfo import summary

To learn more about our model, let's use `torchinfo`'s [`summary()` method](https://github.com/TylerYep/torchinfo#documentation).

To do so, we'll pass in:
 * `model` - the model we'd like to get a summary of.
 * `input_size` - the shape of the data we'd like to pass to our model, for the case of `resnet18`, the input size is `(batch_size, 3, 224, 224)
   
 * `col_names` - the various information columns we'd like to see about our model. 
 * `col_width` - how wide the columns should be for the summary.
 * `row_settings` - what features to show in a row.

In [22]:
# Let us print the model summary and we will see which layers are trainable
# The model expects images to be fed with dimensions - batchsize, channels, height, width
# We can use the col names feature of the summary function to look at what we want (check torchinfo documentation - 
# https://github.com/TylerYep/torchinfo

model_summary = summary(model,(1,3,224,224), col_names=['output_size', 'kernel_size','num_params','trainable'])

print(f"Model Summary shows last 2 Conv layers are not frozen:\n{model_summary}")

Model Summary shows last 2 Conv layers are not frozen:
Layer (type:depth-idx)                   Output Shape              Kernel Shape              Param #                   Trainable
ResNet                                   [1, 4]                    --                        --                        Partial
├─Conv2d: 1-1                            [1, 64, 112, 112]         [7, 7]                    (9,408)                   False
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         --                        (128)                     False
├─ReLU: 1-3                              [1, 64, 112, 112]         --                        --                        --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           3                         --                        --
├─Sequential: 1-5                        [1, 64, 56, 56]           --                        --                        False
│    └─BasicBlock: 2-1                   [1, 64, 56, 56]           -- 

### We can figure out all of the pretrained models using the above techniques ###

Just one word of caution 
- all models do not have the same names for layers and sometimes looping through the layers may be little different.
- Check online or documentation if you have such issues

Let us do one more - vgg16

In [23]:
# Load a vgg16 model with pre-trained default weights
# This model was trained originally on 1000 classes and hence has 1000 outputs

# Get the available weights for vgg16
vgg16_defaultweights = models.VGG16_Weights.DEFAULT

# Get the model and load it with the weights
model_vgg16 = models.vgg16(weights=vgg16_defaultweights) 

print(f"The architecture of the vgg16 model has many convolution layers and a fully connected classifier at the end:\n{model_vgg16}")

The architecture of the vgg16 model has many convolution layers and a fully connected classifier at the end:
VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), pa

In [24]:
# There is a convenient way of getting the transforms that were done on the original images
# We get it from the weights information
vgg16_auto_transforms = vgg16_defaultweights.transforms()
print(f"Originally used vgg16 transforms are:\n{vgg16_auto_transforms}")

Originally used vgg16 transforms are:
ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [25]:
# Now as you can see the vgg16 model has a different architecture and is structured differently also
# We have the sequnetial layer which house all the convolutions and the classifier layer which house all the fc layers

# So, how do we find how many convolution layers are there so that we can freeze the required amount 
# We simply count

conv_count = 0
for name, layer in model_vgg16.named_modules():
    if isinstance(layer, torch.nn.Conv2d):  # look for the name
        conv_count += 1
        print(f"Convolutional layer {conv_count}: {name}")

print(f"\nTotal convolutional layers: {conv_count}")

Convolutional layer 1: features.0
Convolutional layer 2: features.2
Convolutional layer 3: features.5
Convolutional layer 4: features.7
Convolutional layer 5: features.10
Convolutional layer 6: features.12
Convolutional layer 7: features.14
Convolutional layer 8: features.17
Convolutional layer 9: features.19
Convolutional layer 10: features.21
Convolutional layer 11: features.24
Convolutional layer 12: features.26
Convolutional layer 13: features.28

Total convolutional layers: 13


- __Classifier Structure:__ VGG16's classifier is a sequential module (nn.Sequential) containing multiple fully connected layers. You can access and modify individual layers within this module using their index (e.g., classifier[0], classifier[3], classifier[6]).
- __Task-Specific Modification:__ You'll need to adjust the output size of the final layer (5 in this example) to match the number of classes in your specific classification task.

In [26]:
# Let us take the examples of freezing all conv layers
for param in model_vgg16.parameters():
    param.requires_grad = False

# Now, Freeze all layers except the last 3 convolutional layers
for name, param in model_vgg16.named_parameters():
    if "features.24" in name or "features.26" in name or "features.28" in name:  
        param.requires_grad = True
    else:
        param.requires_grad = False

# Modify the final fully connected layer for your task - that is as many output or classes out application will have.
# Suppose we had 4 output
num_classes = 4

# Modify the classifier layer for a 4-class classification task
num_ftrs = model_vgg16.classifier[6].in_features  # Get the number of input features of the last layer
model_vgg16.classifier[6] = nn.Linear(num_ftrs, num_classes)  # Replace the last layer with a new linear layer

# Set the entire classifier layer to requires_grad = True
for param in model_vgg16.classifier.parameters():
    param.requires_grad = True

In [27]:
# let us look at the summary

model_summary = summary(model_vgg16,(1,3,224,224), col_names=['output_size', 'kernel_size','num_params','trainable'])

print(f"Model Summary shows last 3 Conv and classifier layers are not frozen:\n\n{model_summary}")

Model Summary shows last 3 Conv and classifier layers are not frozen:

Layer (type:depth-idx)                   Output Shape              Kernel Shape              Param #                   Trainable
VGG                                      [1, 4]                    --                        --                        Partial
├─Sequential: 1-1                        [1, 512, 7, 7]            --                        --                        Partial
│    └─Conv2d: 2-1                       [1, 64, 224, 224]         [3, 3]                    (1,792)                   False
│    └─ReLU: 2-2                         [1, 64, 224, 224]         --                        --                        --
│    └─Conv2d: 2-3                       [1, 64, 224, 224]         [3, 3]                    (36,928)                  False
│    └─ReLU: 2-4                         [1, 64, 224, 224]         --                        --                        --
│    └─MaxPool2d: 2-5                    [1, 64, 112